# Project on PageRank: Algorithms and Codes
#### D'Amico Davide, Rainone Gerardo

## 1. The General Power Method

#### Helper functions to avoid any libraries

In [ ]:
def vector_scalar_div(vec, scalar):
      return [x / scalar for x in vec]

def vector_norm(vec):
      sq_sum = sum(x**2 for x in vec)
      return sq_sum ** 0.5

def dot_product(v1, v2):
      return sum(x*y for x, y in zip(v1, v2))

def matrix_vector_mult(Mat, vec):
      n_rows = len(Mat)
      n_cols = len(Mat[0])
      result = [0.0] * n_rows
      for i in range(n_rows):
          for j in range(n_cols):
               result[i] += Mat[i][j] * vec[j]
      return result

def vector_scalar_mult(vec, scalar):
      return [x * scalar for x in vec]

def vector_add(v1, v2):
      return [x + y for x, y in zip(v1, v2)]

def vector_subtract(v1, v2):
      return [x - y for x, y in zip(v1, v2)]

def vector_norm_1(vec):
      return sum(abs(x) for x in vec)

#### Power Method Algorithm

In [ ]:
def power_method(A, v, max_iter=100, tol=1e-6):

    # 0. Initial Normalization of vector v(0)
    # Necessary for the Rayleigh quotient to be a valid estimate immediately
    norm_v = vector_norm(v)
    v = vector_scalar_div(v, norm_v)

    # 1. Initialize lambda(0)
    # lambda = v^T * A * v (Rayleigh Quotient with norm 1)
    Av = matrix_vector_mult(A, v)
    lambda_old = dot_product(v, Av)

    m = 0

    # 2. Iterative Loop
    while m < max_iter:
        # v_tilde^(m+1) = A * v^(m)
        v_tilde = matrix_vector_mult(A, v)

        # lambda^(m+1) = v^(m) * v_tilde^(m+1)
        # Since v^(m) is normalized, this is the Rayleigh Quotient
        lambda_new = dot_product(v, v_tilde)

        # v^(m+1) = v_tilde / ||v_tilde||
        norm_v_tilde = vector_norm(v_tilde)

        # Avoiding division by zero
        if norm_v_tilde == 0:
            print("Warning: Zero vector reached.")
            return 0, v

        v = vector_scalar_div(v_tilde, norm_v_tilde)

        # Convergence Check
        # |lambda^(m+1) - lambda^(m)| < tol * |lambda^(m+1)|
        diff = abs(lambda_new - lambda_old)
        abs_lambda = abs(lambda_new)

        if diff < tol * abs_lambda:
            return lambda_new, v

        lambda_old = lambda_new
        m += 1

    return lambda_old, v

## 2. The Google PageRank Algorithm

In [ ]:
def PageRank(A, m, max_iter=100, tol=1e-6):

    def create_ranked_list(vector):
          """
          Function that transforms the score vector into an ordered list
          of (page, score)
          """
          page_scores = list(enumerate(vector, start=1))

          # Sorts the list of tuples
          # item[1] sort based on the score
          ranked_list = sorted(page_scores, key=lambda item: item[1], reverse=True)
          return ranked_list


    # 1. Initialization
    n = len(A)

    # 's' is the vector with all entries 1/n
    s = [1.0 / n] * n

    # x_old is our initial guess
    x_old = list(s)

    # Constant term 'm*s'
    term_ms = vector_scalar_mult(s, m)

    for k in range(max_iter):

        # 2. x_new = (1-m)A*x_old + m*s

        # Part A:  A * x_old
        Ax = matrix_vector_mult(A, x_old)

        # Part B: Multiply by (1-m)
        term_A = vector_scalar_mult(Ax, 1 - m)

        # Part C: Add the term m*s
        x_new = vector_add(term_A, term_ms)

        # 3. Check Convergence
        # We use the L1 norm of the difference vector
        diff_vec = vector_subtract(x_new, x_old)
        error = vector_norm_1(diff_vec)

        if error < tol:
            # Convergence reached
            print(f"PageRank converged in {k+1} iterations.")
            return create_ranked_list(x_new)

        x_old = x_new

    return create_ranked_list(x_old)

## 3. Testing the PageRank algorithm on the two paper's figures (**Figures 2.1,2.2**)

In [ ]:
if __name__ == "__main__":

    # Figure 2.1 (Connected Web)
    A_fig21 = [
        [0.0,   0.0, 1.0, 0.5],
        [1/3,   0.0, 0.0, 0.0],
        [1/3,   0.5, 0.0, 0.5],
        [1/3,   0.5, 0.0, 0.0]
    ]

    print(" TESTING FIGURE 2.1 (Connected Web)")
    ranks_21 = PageRank(A_fig21, m=0.15)

    for rank, (page, score) in enumerate(ranks_21, 1):
        print(f"Rank {rank}: Page {page} (Score: {score:.4f})")



    # Figure 2.2 (Disconnected Web)
    A_fig22 = [
        [0, 1, 0, 0, 0],
        [1, 0, 0, 0, 0],
        [0, 0, 0, 1, 0.5],
        [0, 0, 1, 0, 0.5],
        [0, 0, 0, 0, 0]
    ]

    # Fig 2.2 technically has no "dangling nodes".
    # Page 5 has 0 backlinks (row 5 is 0), but it links to pages 3 and 4,
    # so its column sum is 1. We do not need a fix for this specific matrix.

    print("\n TESTING FIGURE 2.2 (Disconnected Web)")
    ranks_22 = PageRank(A_fig22, m=0.15)

    for rank, (page, score) in enumerate(ranks_22, 1):
        print(f"Rank {rank}: Page {page} (Score: {score:.4f})")

 TESTING FIGURE 2.1 (Connected Web)
PageRank converged in 19 iterations.
Rank 1: Page 1 (Score: 0.3682)
Rank 2: Page 3 (Score: 0.2880)
Rank 3: Page 4 (Score: 0.2021)
Rank 4: Page 2 (Score: 0.1418)

 TESTING FIGURE 2.2 (Disconnected Web)
PageRank converged in 2 iterations.
Rank 1: Page 3 (Score: 0.2850)
Rank 2: Page 4 (Score: 0.2850)
Rank 3: Page 1 (Score: 0.2000)
Rank 4: Page 2 (Score: 0.2000)
Rank 5: Page 5 (Score: 0.0300)


## 4. Processing the "Hollins" Dataset

In [ ]:
def load_and_process_data(filename):


    with open(filename, 'r', encoding='utf-8', errors='ignore') as f:
        lines = f.readlines()


    # 1. Read Metadata (First line)
    if not lines: return [], {}

    meta = lines[0].split()
    num_pages = int(meta[0])

    urls = {}
    links = []

    # 2. File Parsing (Distinguish between URLs and Links)
    reading_urls = True

    for line in lines[1:]:
        parts = line.split()
        if not parts: continue

        # Logic to separate URLs from Links
        if reading_urls:
            # Check if line looks like a URL definition
            if len(parts) >= 2 and ("http" in parts[1] or not parts[1].isdigit()):
                try:
                    page_id = int(parts[0])
                    url = parts[1]
                    urls[page_id] = url
                except ValueError:
                    reading_urls = False # Failed to parse as URL, assume Link section started
            else:
                reading_urls = False # Numeric line found, assume Link section started

        if not reading_urls:
            # Link Section: Source_ID Destination_ID
            if len(parts) >= 2:
                try:
                    src = int(parts[0])
                    dst = int(parts[1])
                    links.append((src, dst))
                except ValueError:
                    continue

    print(f"Parsed {len(urls)} URLs and {len(links)} links.")

    # 3. Matrix A Construction
    # Initialize with 0.0
    A = [[0.0] * num_pages for _ in range(num_pages)]

    # Array to track outgoing links (to calculate probabilities)
    # Size num_pages + 1 because file indices are 1-based
    out_degree = [0] * (num_pages + 1)

    # 1: Count outgoing links
    for src, dst in links:
        if src <= num_pages and dst <= num_pages:
            out_degree[src] += 1

    # 2: Fill probabilities for existing links
    for src, dst in links:
        if src <= num_pages and dst <= num_pages:
            # Convert to 0-based index
            row = dst - 1
            col = src - 1
            if out_degree[src] > 0:
                A[row][col] = 1.0 / out_degree[src]

    # Dangling Nodes Fix
    # We replace zero columns with 1/n to make A strictly column-stochastic
    dangling_count = 0
    fill_value = 1.0 / num_pages

    for col_idx in range(num_pages):
        if out_degree[col_idx + 1] == 0:
            dangling_count += 1
            for row_idx in range(num_pages):
                A[row_idx][col_idx] = fill_value

    print(f"Corrected {dangling_count} dangling nodes.")

    return A, urls

#### Testing on the processed dataset

In [ ]:
if __name__ == "__main__":
    matrix_A, map_urls = load_and_process_data('hollins.dat')

    if matrix_A:
        print("\nCalculating PageRank...")
        final_ranking = PageRank(matrix_A, m=0.15, max_iter=100, tol=1e-6)

        print("TOP 10 WEBPAGES (PageRank)")

        for i in range(min(10, len(final_ranking))):
            page_idx, score = final_ranking[i]
            # Retrieve URLs
            url = map_urls.get(page_idx, "URL Unknown")

            print(f"Rank {i+1:2d}: [Score: {score:.6f}] - ID: {page_idx} - {url}")

        total_score = sum(score for _, score in final_ranking)
        print("\nVerification check:")
        print(f"Sum of all PageRank scores: {total_score:.6f}")

Parsed 6012 URLs and 23875 links.
Corrected 3189 dangling nodes.

Calculating PageRank...
PageRank converged in 58 iterations.
TOP 10 WEBPAGES (PageRank)
Rank  1: [Score: 0.019879] - ID: 2 - http://www.hollins.edu/
Rank  2: [Score: 0.009288] - ID: 37 - http://www.hollins.edu/admissions/visit/visit.htm
Rank  3: [Score: 0.008610] - ID: 38 - http://www.hollins.edu/about/about_tour.htm
Rank  4: [Score: 0.008065] - ID: 61 - http://www.hollins.edu/htdig/index.html
Rank  5: [Score: 0.008027] - ID: 52 - http://www.hollins.edu/admissions/info-request/info-request.cfm
Rank  6: [Score: 0.007165] - ID: 43 - http://www.hollins.edu/admissions/apply/apply.htm
Rank  7: [Score: 0.006583] - ID: 425 - http://www.hollins.edu/academics/library/resources/web_linx.htm
Rank  8: [Score: 0.005989] - ID: 27 - http://www.hollins.edu/admissions/admissions.htm
Rank  9: [Score: 0.005572] - ID: 28 - http://www.hollins.edu/academics/academics.htm
Rank 10: [Score: 0.004452] - ID: 4023 - http://www1.hollins.edu/faculty/